# LLM-as-Judge — AI Engineering Interview Prep

Using Claude to evaluate LLM outputs: scoring, ranking, and critique patterns.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import anthropic
import os
import json

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"

def judge(prompt, max_tokens=400):
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.content[0].text

print(f"Judge model: {MODEL}")

## 1. Pointwise Scoring

In [ ]:
# Pointwise: score a single response on a rubric
question = "What is the difference between precision and recall?"

responses = {
    "Response A": "Precision measures how many of your positive predictions were correct. Recall measures how many actual positives you caught. Use precision when false positives are costly (spam filter), recall when false negatives are costly (cancer detection).",
    "Response B": "Precision = TP/(TP+FP), Recall = TP/(TP+FN). They trade off against each other.",
    "Response C": "Precision is about being right when you say yes. Recall is about finding all the yes cases. Think of it like a fishing net: precision = few non-fish caught, recall = few fish escaped."
}

JUDGE_RUBRIC = """Score this ML answer on a scale of 1-5 for each criterion.
Output ONLY valid JSON: {"accuracy": N, "completeness": N, "clarity": N, "examples": N, "overall": N, "rationale": "..."}

Criteria:
- accuracy (1-5): Is the definition technically correct?
- completeness (1-5): Does it cover both concepts adequately?
- clarity (1-5): Is it easy to understand?
- examples (1-5): Does it give useful examples or context?

Question: {question}
Answer: {answer}"""

scores = {}
for name, answer in responses.items():
    prompt = JUDGE_RUBRIC.format(question=question, answer=answer)
    raw = judge(prompt)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        scores[name] = json.loads(clean)
    except json.JSONDecodeError:
        scores[name] = {"error": raw[:100]}
    print(f"{name}: {scores[name]}")

# Find winner
winner = max(scores.items(), key=lambda x: x[1].get("overall", 0))
print(f"\nBest response: {winner[0]} (overall={winner[1].get('overall', '?')})")

## 2. Pairwise Comparison

In [ ]:
# Pairwise: compare two responses head-to-head
pairwise_prompt = """You are an impartial judge evaluating two ML explanations.
Output ONLY valid JSON: {"winner": "A" or "B" or "tie", "reasoning": "...", "scores": {"A": N, "B": N}}

Question: {question}

[Response A]
{a}

[Response B]
{b}

Which response better explains the concept to an ML engineer? Judge on: accuracy, clarity, practical value."""

response_names = list(responses.keys())
response_vals = list(responses.values())

# A vs B, B vs C
pairs = [(0, 1), (1, 2)]
for i, j in pairs:
    prompt = pairwise_prompt.format(
        question=question,
        a=response_vals[i],
        b=response_vals[j]
    )
    raw = judge(prompt)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        result = json.loads(clean)
        actual_winner = response_names[i] if result.get("winner") == "A" else (
            response_names[j] if result.get("winner") == "B" else "Tie"
        )
        print(f"{response_names[i]} vs {response_names[j]} → Winner: {actual_winner}")
        print(f"  Reasoning: {result.get('reasoning', '')[:100]}")
    except json.JSONDecodeError:
        print(f"Parse error: {raw[:100]}")

## 3. Critique and Revision Loop

In [ ]:
# Generate → Critique → Revise loop
task = "Write a 2-sentence explanation of attention mechanisms in transformers for a job interview."

# Round 1: Generate initial response
initial = judge(task, max_tokens=200)
print("Initial response:")
print(initial)

# Round 2: Critique
critique_prompt = f"""Critique this explanation. Be specific about what's missing, unclear, or inaccurate.
Output: ISSUES: [list] | MISSING: [list] | SUGGESTED_IMPROVEMENT: [one sentence]

Explanation: {initial}"""

critique = judge(critique_prompt, max_tokens=300)
print("\nCritique:")
print(critique)

# Round 3: Revise based on critique
revise_prompt = f"""Rewrite this explanation addressing the critique. Keep it to 2-3 sentences.

Original: {initial}
Critique: {critique}

Improved version:"""

revised = judge(revise_prompt, max_tokens=200)
print("\nRevised response:")
print(revised)

## 4. Reference-Free Factual Checking

In [ ]:
# Check factual claims without reference documents
claims = [
    "BERT uses a decoder-only transformer architecture.",
    "Dropout randomly zeros activations during training to reduce overfitting.",
    "The Adam optimizer combines SGD with momentum and RMSProp.",
    "GPT models are trained with masked language modeling like BERT.",
    "Batch normalization normalizes across the batch dimension for each feature."
]

fact_check_prompt = """Check if this ML claim is correct. Output ONLY JSON:
{"verdict": "correct" or "incorrect" or "partially_correct", "explanation": "...", "correction": "..." or null}

Claim: {claim}"""

for claim in claims:
    raw = judge(fact_check_prompt.format(claim=claim), max_tokens=200)
    clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        result = json.loads(clean)
        verdict = result.get("verdict", "?")
        marker = "OK" if verdict == "correct" else ("WRONG" if verdict == "incorrect" else "PARTIAL")
        print(f"[{marker}] {claim[:60]}...")
        if result.get("correction"):
            print(f"       Correction: {result['correction'][:80]}")
    except json.JSONDecodeError:
        print(f"Claim: {claim[:50]} → {raw[:80]}")

## Interview Tips

- **LLM-as-judge bias**: models favor verbose, confident-sounding answers. Mitigate with rubrics.
- **Position bias**: LLMs prefer the first response in pairwise comparisons. Always swap A/B and average.
- **Calibration**: compare judge scores to human labels — high correlation = reliable judge.
- **Pointwise vs pairwise**: pointwise is cheaper (1 call/response); pairwise is more reliable.
- **Reference-based eval**: BLEU/ROUGE for text overlap; LLM-judge for semantic quality.
- **Self-critique**: same model generating and judging leads to sycophancy — use a stronger judge model.